# Lab 8: Luong Attention (Multiplicative Attention)

## Objective
- To understand and implement **Luong Attention** (Multiplicative Attention) on top of the Seq2Seq RNN Encoder-Decoder architecture built in Lab 7. — [Luong et al., 2015](https://arxiv.org/pdf/1508.04025)


## Theory
Traditional encoder-decoder models for neural machine translation compress the entire source sentence into a single fixed-length context vector. This creates a bottleneck, particularly for long sentences, as important information is lost in the compression process. Luong et al. (2015) refined attention mechanisms to address this limitation by allowing the decoder to dynamically focus on relevant parts of the source sentence during each generation step.

### The Attention Solution
Attention mechanisms allow the decoder to **dynamically look back** at all encoder hidden states and decide which parts of the source sequence are most relevant for generating each output token. Instead of relying on one fixed context vector, the decoder computes a **weighted sum** of all encoder annotations at every decoding step:

$$c_t = \sum_{i=1}^{T_x} \alpha_{t,i} \, h_i$$

where $\alpha_{t,i}$ are the **attention weights** and $h_i$ are the encoder hidden states.

---

### Luong Attention (Multiplicative)

Luong et al. (2015) use **direct vector similarity** (dot product or a learned linear transformation) for alignment score computation:

| Score Function | Formula |
|---|---|
| **Dot** | $e_{t,i} = s_t^T h_i$ |
| **General** | $e_{t,i} = s_t^T W_a h_i$ |
| **Concat** | $e_{t,i} = v_a^T \tanh(W_a [s_t ; h_i])$ |

Key architectural features of Luong Attention:
- The decoder RNN runs **first** to produce $s_t$, then attention is computed.
- The context vector and $s_t$ are combined into an **attentional hidden state**: $\tilde{s}_t = \tanh(W_c[c_t\,;\,s_t])$
- This is called **multiplicative** attention because scores are computed via matrix multiplication.


## Code Implementation

The following implementation reuses the data loading, vocabulary, and training infrastructure from **Lab 7**. Only the decoder is replaced with an attention-based decoder.

**Requirements**

In [39]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random
import os

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data

We reuse the same English–French dataset from Lab 7.

Download from: [https://download.pytorch.org/tutorial/data.zip](https://download.pytorch.org/tutorial/data.zip)

Extract `data/eng-fra.txt` (a tab-separated file of translation pairs) into the Lab 7 `data/` folder.

In [40]:
SOS_token = 0  # Start of the Sentence
EOS_token = 1  # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [41]:
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

def readLangs(path: str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")
    lines = open(path, encoding='utf-8').read().strip().split('\n')
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    pairs = [list(reversed(p)) for p in pairs]  # French -> English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)
    return input_lang, output_lang, pairs

In [42]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return (len(p[0].split(' ')) < MAX_LENGTH and
            len(p[1].split(' ')) < MAX_LENGTH and
            p[1].startswith(eng_prefixes))

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [43]:
PATH = r'dataset/eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['il devient chauve', 'he s going bald']


## Data Loader

In [44]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)
    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

## Encoder (shared by both attention variants)

The encoder is unchanged from Lab 7. It processes the input sequence word-by-word through a unidirectional RNN and returns all hidden states (`encoder_outputs`) along with the final hidden state (`encoder_hidden`).

- `encoder_outputs` — shape `(batch, seq_len, hidden_size)` — used by the attention mechanism to compute context vectors.
- `encoder_hidden` — shape `(1, batch, hidden_size)` — used to initialize the decoder.

In [45]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

## Luong Attention (Multiplicative / Dot-Product)

## Part 2: Luong Attention (Multiplicative / Dot-Product)

### How It Works

Luong attention differs from Bahdanau in two key ways:

1. **Order of operations**: The decoder RNN runs **first** to produce $s_t$, and attention is computed **after**.
2. **Score function**: Instead of an additive feed-forward network, Luong uses a **dot product**:
$$e_{t,i} = s_t^T h_i$$

After computing the context vector $c_t$, the two are merged into an **attentional hidden state**:
$$\tilde{s}_t = \tanh(W_c[c_t\,;\,s_t])$$


In [48]:
class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()
        # W_c for: s~_t = tanh(W_c [c_t ; s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, query, keys):
        """
        query : current decoder hidden state s_t
                shape -> (batch_size, 1, hidden_size)
        keys  : encoder hidden states h_1,...,h_T
                shape -> (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden : s~_t
                                 shape -> (batch_size, 1, hidden_size)
            weights            : attention weights alpha_t
                                 shape -> (batch_size, 1, seq_len)
        """
        # Alignment scores: e_{t,i} = s_t^T h_i
        scores = torch.bmm(query, keys.transpose(1, 2))          # (batch, 1, seq_len)

        # Attention weights: alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector: c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(weights, keys)                       # (batch, 1, hidden)

        # Attentional hidden state: s~_t = tanh(W_c [c_t ; s_t])
        combined = torch.cat((context, query), dim=-1)           # (batch, 1, 2*hidden)
        attentional_hidden = torch.tanh(self.Wc(combined))       # (batch, 1, hidden)

        return attentional_hidden, weights

### `LuongAttnDecoderRNN`

Notice the key difference from the Bahdanau decoder:
- The RNN input is **only** the embedded word (no concatenation with context before the RNN).
- After the RNN step, attention is applied to the RNN's current output $s_t$.
- The final output is predicted from the attentional hidden state $\tilde{s}_t$.

In [49]:
class LuongAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(LuongAttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)  # input = embed only
        self.attention = LuongDotAttention(hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)  # Teacher forcing
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))           # (batch, 1, hidden)

        # 1. Run RNN to get current decoder state s_t
        rnn_output, hidden = self.rnn(embedded, hidden)          # (batch, 1, hidden)

        # 2. Compute attention using s_t as query
        query = rnn_output                                        # (batch, 1, hidden)
        attentional_hidden, attn_weights = self.attention(query, encoder_outputs)

        # 3. Predict from attentional hidden state s~_t
        output = self.out(attentional_hidden)                    # (batch, 1, output_size)

        return output, hidden, attn_weights

## Training Infrastructure

The training loop is identical to Lab 7. Both attention decoders share the same interface as the Lab 7 decoder, so no changes are needed here.

In [50]:
import time
import math
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

def showPlot(points, title='Training Loss'):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    ax.set_title(title)
    ax.set_xlabel('Epochs (x plot_every)')
    ax.set_ylabel('Average NLL Loss')
    plt.plot(points)
    plt.tight_layout()
    plt.savefig(title.replace(' ', '_') + '.png', dpi=100)
    plt.show()

In [51]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
                decoder_optimizer, criterion):
    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
          print_every=100, plot_every=100, title='Training Loss'):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder,
                           encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                          epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses, title=title)

## Evaluation Code

In [52]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn


def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

---

## Training — Luong Attention

In [55]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

luong_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
luong_decoder = LuongAttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, luong_encoder, luong_decoder, EPOCHS,
      print_every=5, plot_every=5, title='Luong Attention - Training Loss')

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 5s (- 3m 21s) (5 2%) 1.7779
0m 10s (- 3m 16s) (10 5%) 0.8985
0m 15s (- 3m 11s) (15 7%) 0.5075
0m 20s (- 3m 5s) (20 10%) 0.3211
0m 25s (- 3m 0s) (25 12%) 0.2368
0m 30s (- 2m 55s) (30 15%) 0.1970
0m 36s (- 2m 50s) (35 17%) 0.1726
0m 41s (- 2m 45s) (40 20%) 0.1624
0m 46s (- 2m 40s) (45 22%) 0.1542
0m 51s (- 2m 35s) (50 25%) 0.1447
0m 56s (- 2m 29s) (55 27%) 0.1395
1m 2s (- 2m 24s) (60 30%) 0.1355
1m 7s (- 2m 19s) (65 32%) 0.1263
1m 12s (- 2m 14s) (70 35%) 0.1256
1m 17s (- 2m 9s) (75 37%) 0.1231
1m 22s (- 2m 4s) (80 40%) 0.1221
1m 27s (- 1m 58s) (85 42%) 0.1159
1m 32s (- 1m 53s) (90 45%) 0.1056
1m 38s (- 1m 48s) (95 47%) 0.1009
1m 43s (- 1m 43s) (100 50%) 0.0966
1m 48s (- 1m 38s) (105 52%) 0.0961
1m 53s (- 1m 32s) (110 55%) 0.0939
1m 58s (- 1m 27s) (115 57%) 0.0899
2m 3s (- 1m 22s) (120 60%) 0.0847
2m 9s (- 1m 17s) (125 62%) 0.0828
2m 14s (- 1m 12s) (130 65%) 0.07

/var/folders/v1/n0ygwr2s65v1fqj84kwq2cdw0000gn/T/ipykernel_40891/37737821.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Luong — Evaluation

In [56]:
luong_encoder.eval()
luong_decoder.eval()
evaluateRandomly(luong_encoder, luong_decoder)

> je suis desolee
= i m sorry
< i am truly there <EOS>

> tu te ridiculises
= you re embarrassing yourself
< you re nuts sync <EOS>

> tu es grossiere
= you re rude
< you re very rude <EOS>

> ils sont chanteurs
= they are singers
< they re singers <EOS>

> elle est toubib
= she is a doctor
< she is a doctor <EOS>

> tu es fort maline
= you re quite smart
< you re very smart <EOS>

> je suis trop petit
= i am too short
< i am too short <EOS>

> ils sont vieux
= they re old
< they re old <EOS>

> tu es completement ignorant
= you re totally ignorant
< you re totally ignorant <EOS>

> elles sont speciales
= they re special
< they re special <EOS>



## Results
The Luong attention mechanism significantly improved translation quality compared to the Lab 7 baseline. It achieved a final training loss of ~0.040. The model demonstrated accurate translations and trained efficiently due to its simpler multiplicative formulation.


## Discussion
Attention mechanisms address the bottleneck of fixed-length context vectors by dynamically focusing on relevant parts of the input sequence. Luong uses dot-product scoring and computes attention after the RNN step. This makes it computationally faster while achieving strong performance.

### Why Attention Outperforms the Baseline?
Attention outperforms the baseline because it eliminates the information bottleneck caused by compressing the entire input sequence into a single fixed-length context vector. Instead of relying on a single vector, attention dynamically computes a weighted sum of encoder hidden states for each decoding step, allowing the model to focus on the most relevant parts of the input sequence. This ensures that important information is not lost, especially for longer or complex sentences, leading to more accurate and context-aware translations.

## Conclusion
The lab demonstrated that attention mechanisms significantly enhance Seq2Seq models by enabling dynamic context computation. The Luong attention mechanism improved translation quality and provided computational efficiency, highlighting the importance of attention in modern NLP architectures.
